In [0]:
from common_utils.scd_functions import scd_type1
from common_utils.generic_functions import rename_columns, drop_columns , casting
from common_utils.logging import get_logger
import pyspark.sql.functions as f
import json
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    IntegerType,
    ArrayType
)

# ============================================================
# 0. LOGGER
# ============================================================
logger = get_logger("b2s_sales_orders.ipynb")
logger.info("session started")

# =========================================
# 1. CONFIG PATH
# =========================================
dbutils.widgets.text("path", "")
config_path =dbutils.widgets.get("path") 
print(config_path)
logger.info("setting the configurations")


# =========================================
# 2. Read Config
# =========================================
with open(config_path, "r") as file:
  config = json.load(file) 

# =========================================
# 3. CONFIG Sections
# =========================================
source_table = config['source']["source_table"]
source_name = config["source"]["source_name"]
#target_table = config["target"]["target_table"]



logger.info("Configuration has been done")
# ============================================================
# 0. Schemas
# ============================================================

# Schema for promotion information
promotion_schema = StructType([
    StructField("promo_disc", DoubleType(), True),
    StructField("promo_id", StringType(), True),
    StructField("promo_item", StringType(), True),
    StructField("promo_qty", StringType(), True)
])
# Schema for product information

product_schema = StructType([
    StructField("id", StringType()),
    StructField("name", StringType()),
    StructField("price", StringType()),
    StructField("curr", StringType()),
    StructField("qty", StringType()),
    StructField("unit", StringType())
])

# sales order schema this gonna be the overall schema and we are using the promotion schema and product schema inside this schema when we parse the data
sales_order_schema = StructType([
    StructField("order_number", StringType()),
    StructField("customer_id", StringType()),
    StructField("customer_name", StringType()),
    StructField("number_of_line_items", StringType()),

    # Handles values such as 1.564627663E9
    StructField("order_datetime", DoubleType()),

    StructField(
        "ordered_products",
        ArrayType(product_schema)
    ),

    StructField(
        "promo_info",
        ArrayType(promotion_schema)
    ),

    StructField(
        "clicked_items",
        ArrayType(
            ArrayType(StringType())
        )
    )
])
df = spark.read.table("retail_data_platform.bronze_layer.cosmosdb_sales_ingestion")
# ============================================================
# 6. PARSE JSON DATA
# ============================================================
def parse_json_data(df):
 logger.info("paring the json data")
 df = df.withColumn("data", f.from_json(df["json_data"],sales_order_schema)).select("data.*")
 logger.info("paring finished for source[%s]",source_name)
 return df
# ============================================================
# 7. SALES ORDERS
# ============================================================
def transformations_sales_order(df,config):
    logger.info(
        "[%s] Transforming sales_orders",
        source_name
    )
    df = rename_columns(df,config["tables"]["sales_orders"].get("rename_columns",{}))
    df = df.withColumn("order_timestamp",f.from_unixtime(f.col("order_datetime").cast("long")).cast("timestamp")).select(
            "order_number",
            "customer_id",
            "customer_name",
            "number_of_line_items",
            "order_timestamp"
        )
    df = casting(df,config["tables"]["sales_orders"].get("cast_columns",{}))
    df=df.select(*config["tables"]["sales_orders"].get("select_columns",[]))
    logger.info(
        "[%s] sales_orders transformation completed",
        source_name
    )
    df = df.dropDuplicates()
    return df

def transformations_order_products(df,config):
    logger.info(
        "[%s] Transforming order_products",
        source_name
    )
   
  
  

# ============================================================
# 8. ORDER PRODUCTS
# ============================================================
  
    df = df.withColumn("order_products",f.explode_outer(f.col("ordered_products"))).select(
    "customer_id",
    "customer_name",
    "order_number",
    f.col("order_products.id").alias("id"),
    f.col("order_products.name").alias("product_name"),
    f.col("order_products.price").alias("price"),
    f.col("order_products.curr").alias("curr"),
    f.col("order_products.qty").alias("qty"),
    f.col("order_products.unit").alias("unit")
    ).filter(f.col("id").isNotNull())
                   #renaming columns

    df = rename_columns(df,config["tables"]["order_products"].get("rename_columns",{}))

    #casting columns
    df = casting(df,config["tables"]["order_products"].get("cast_columns",{}))
    df = df.select(*config["tables"]["order_products"].get("select_columns",[]))
    logger.info(
        "[%s] order_products transformation completed",
        source_name
    )
    df = df.dropDuplicates()
    return df
    

# ============================================================
# 9. PROMOTIONS
# ============================================================

def transformations_promotions(df,config):
     logger.info(
        "[%s] Transforming promotions",
        source_name
    )
     
     df = df.withColumn(
    "promotion", 
    f.explode_outer(f.col("promo_info"))
    ).select(
    "Customer_id",
    "order_number",
    f.col("promotion.promo_id").alias("promo_id"),
    f.col("promotion.promo_item").alias("promo_product_id"),
    f.col("promotion.promo_disc").alias("discount"),
    f.col("promotion.promo_qty").alias("promo_quantity")
     ).filter(
     f.col("promo_id").isNotNull())

     df = rename_columns(df,config["tables"]["promotions"].get("rename_columns",{}))

     df = casting(df,config["tables"]["promotions"].get("cast_columns",{}))
     df = df.select(*config["tables"]["promotions"].get("select_columns",{}))
     
     logger.info(
        "[%s] promotions transformation completed",
        source_name
    )
     df =  df.dropDuplicates()
     return df

# ============================================================
# 10. CLICKED ITEMS
# ============================================================

def transform_clicked_items(df, config):

    logger.info(
        "[%s] Transforming clicked_items",
        source_name
    )

    df = (
        df
        .withColumn(
            "clicked_item",
            f.explode_outer(
                f.col("clicked_items")
            )
        )
        .select(
            "customer_id",
            "customer_name",
            "order_number",
            f.col("clicked_item")
            .getItem(0)
            .alias("product_id"),
            f.col("clicked_item")
            .getItem(1)
            .alias("score")
        )
        .filter(
          f.col("product_id").isNotNull()
        )
    )

    df = casting(
        df,
        config["tables"]["clicked_items"].get(
            "cast_columns",
            {}
        )
    )

    df = df.select(
        *config["tables"]["clicked_items"]["select_columns"]
    )

    logger.info(
        "[%s] clicked_items transformation completed",
        source_name
    )

    df = df.dropDuplicates()

    return df
# ============================================================
# 11. READ BRONZE
# ============================================================


logger.info(
    "[%s] Reading from Bronze table %s",
    source_name,
    source_table
)

df = spark.read.table(source_table)

logger.info(
    "[%s] Read %d records from Bronze table %s",
    source_name,
    df.count(),
    source_table
)


# ============================================================
# 12. PARSE NESTED JSON
# ============================================================

df = parse_json_data(df)

# ============================================================
# 13. TRANSFORMATIONS FOUR SILVER TABLES
# ============================================================
sales_order = transformations_sales_order(df,config)
promotions = transformations_promotions(df,config)
order_products = transformations_order_products(df,config)
clicked_items = transform_clicked_items(df,config)

tables = {
    "sales_orders": sales_order,
    "order_products": order_products,
    "promotions": promotions,
    "clicked_items": clicked_items
}
for table_name, transformed_df in tables.items():
    table_config = config["tables"][table_name]
    target_table = table_config["target_table"]
    scd_config = table_config.get("scd",{})

    logger.info(
        "[%s] Processing target table %s",
        table_name,
        target_table
    )
     # --------------------------------------------------------
    # Target exists
    # --------------------------------------------------------
    if spark.catalog.tableExists(target_table):
        if scd_config.get("type")==1:
            logger.info(
                "[%s] Starting SCD Type 1",
                table_name
            )

            key_columns = scd_config.get("key_columns",[])
            transformed_df = transformed_df.dropDuplicates(key_columns)
            scd_type1(spark , transformed_df,target_table,key_columns)
        else:
            raise ValueError(
                f"Unsupported SCD type "
                f"[{scd_config.get('type')}] "
                f"for table [{table_name}]"
            )
     # --------------------------------------------------------
    # Initial load
    # --------------------------------------------------------
    else:
        logger.info(
            "[%s] Target does not exist - creating table",
            table_name
        )
        spark.sql("""
            CREATE SCHEMA IF NOT EXISTS
            retail_data_platform.silver
        """)

        (
            transformed_df
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )

        logger.info(
            "[%s] Silver table created successfully",
            table_name
        )


# ============================================================
# 15. COMPLETION
# ============================================================

logger.info(
    "[%s] All Silver tables processed successfully",
    source_name
)